# Survival analysis

`survival::lung` holds 228 patients from a North Central Cancer Treatment Group trial,
with survival time in days and a censoring flag — some patients were still alive when the
study ended, so their true survival time is only known to exceed what was observed.
Discarding or ignoring censored rows biases everything; survival methods use them.

The numbers below are checked against the values this dataset is known for.

In [ ]:
library(survival)
library(survminer)

data(cancer, package = "survival")
str(lung[, c("time", "status", "sex", "age", "ph.ecog")])

## Kaplan-Meier by sex

`Surv(time, status == 2)` marks who died (status 2) versus who was censored. The
Kaplan-Meier estimator turns that into a survival curve without assuming a distribution.

In [ ]:
fit <- survfit(Surv(time, status == 2) ~ sex, data = lung)
fit

In [ ]:
med <- summary(fit)$table[, "median"]
cat(sprintf("median survival: male %.0f days, female %.0f days\n", med[1], med[2]))
stopifnot(med[1] == 270, med[2] == 426)

In [ ]:
ggsurvplot(fit, data = lung, conf.int = TRUE, risk.table = TRUE,
           legend.labs = c("male", "female"), xlab = "Days",
           palette = c("#3b6ea5", "#a53b6e"))

## Is the difference real?

The log-rank test compares the two curves over the whole follow-up rather than at one
arbitrary time point.

In [ ]:
lr <- survdiff(Surv(time, status == 2) ~ sex, data = lung)
lr

In [ ]:
p <- pchisq(lr$chisq, df = 1, lower.tail = FALSE)
cat(sprintf("log-rank chi-square = %.2f, p = %.5f\n", lr$chisq, p))
stopifnot(abs(lr$chisq - 10.33) < 0.05, p < 0.002)

## Cox proportional hazards

The log-rank test says *whether* the curves differ; a Cox model says by how much, and
adjusts for other covariates at the same time. The hazard ratio for female patients is
about 0.59 — roughly a 41% lower hazard at any given moment.

In [ ]:
cox <- coxph(Surv(time, status == 2) ~ sex, data = lung)
summary(cox)

In [ ]:
hr <- exp(coef(cox))[["sex"]]
cat(sprintf("hazard ratio (female vs male) = %.3f\n", hr))
stopifnot(abs(hr - 0.588) < 0.005)

## Adjusting, and checking the assumption

A Cox model assumes hazards stay proportional over time. `cox.zph()` tests that; a small
p-value means the effect changes with time and the single hazard ratio is misleading.

In [ ]:
cox_adj <- coxph(Surv(time, status == 2) ~ sex + age + ph.ecog, data = lung)
summary(cox_adj)$coefficients

In [ ]:
zph <- cox.zph(cox_adj)
zph

In [ ]:
cat(sprintf("global proportional-hazards p = %.3f\n", zph$table["GLOBAL", "p"]))
stopifnot(zph$table["GLOBAL", "p"] > 0.05)
cat("proportional hazards not rejected\n")